# SleepStageNet CNN+BiLSTM — Colab Training

Run 20-fold LOSO-CV on Colab T4 GPU.

**Runtime:** Select **GPU (T4)** kernel before running: Select Kernel → Colab → New Colab Server → GPU → T4

In [ ]:
# 1. Configuration — edit these paths/settings as needed
import os

# Where to clone the repo (Colab ephemeral storage is fine — code is lightweight)
REPO_DIR = "/content/deepsleepnet-lite"

# OUTPUT_DIR points to Google Drive — not Colab's ephemeral /content storage.
# This is critical: Colab sessions disconnect after ~12 hours (or sooner on free tier).
# With output on Drive, completed folds survive crashes. Combined with --skip_existing,
# you can simply reconnect and re-run the training cell to resume from where it left off.
OUTPUT_DIR = "/content/drive/MyDrive/University-of-Washington/Winter-2026/SleepStageNet/temporal_output"

# Training hyperparameters
SEQ_LEN = 5         # Sequence length (should be odd: 3, 5, 11, 21)
BATCH_SIZE = 64      # Batch size
FOLDS = range(20)    # Which folds to train (range(20) for full LOSO-CV)

# Data download (Google Drive file ID for the preprocessed Sleep-EDF zip)
DATA_FILE_ID = "1wDu9tl6_P250k522tQC9LUUVh7ocG1_x"
DATA_DIR = f"{REPO_DIR}/data/SleepEDF/processed/eeg_FpzCz_PzOz_v1"

# Repo URL
REPO_URL = "https://github.com/manishdas/deepsleepnet-lite.git"

In [ ]:
# 2. Mount Drive, install deps, clone repo
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-learn gdown

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")

print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# 3. Download data (skip if already exists)
import glob

eeg_files = glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz'))

if len(eeg_files) >= 39:
    print(f"Data already exists: {len(eeg_files)} files")
else:
    !gdown {DATA_FILE_ID} -O /tmp/data.zip
    !cd {REPO_DIR} && unzip -q -o /tmp/data.zip
    eeg_files = glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz'))
    print(f"Downloaded: {len(eeg_files)} files")

In [ ]:
# 4. Verify GPU
import torch, multiprocessing
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'CPU cores: {multiprocessing.cpu_count()}')

In [ ]:
# 5. Sanity check — 2 epochs per stage (quick validation before full run)
%cd {REPO_DIR}/temporal
!python train_sequence.py --fold 0 --seq_len {SEQ_LEN} --batch_size {BATCH_SIZE} \
    --cnn_epochs 2 --lstm_epochs 2 --finetune_epochs 2 \
    --output_dir {OUTPUT_DIR}/sanity_check

In [ ]:
# 6. Full training (saves directly to Drive, skip_existing for crash recovery)
for fold in FOLDS:
    !python train_sequence.py --fold {fold} --seq_len {SEQ_LEN} --batch_size {BATCH_SIZE} \
        --output_dir {OUTPUT_DIR} --skip_existing

In [ ]:
# 7. Verify results on Drive
!ls {OUTPUT_DIR}/fold*_results.json | wc -l
!ls {OUTPUT_DIR}/*.pt | wc -l